# Rede neural convolucional com transformada Wavelet no pipeline 


## 1. Dataset - MiniMIAS

### 1.1 Rodar localmente o programa

- Para isso é necessário baixar e extrair o dataset para a máquina local. 

**Link:** https://www.repository.cam.ac.uk/items/b6a97f0c-3b9b-40ad-8f18-3d121eef1459

- Baixe o dataset em Files;
- Após extrair em uma pasta os arquivos. Comando via terminal: _unzip MIAS-DB.zip_

In [5]:
import os

pasta = "dataset-mias/"

arquivos = os.listdir(pasta)
contador = 0

for arquivo in arquivos:
    if ".pgm" in arquivo:
        contador += 1

print(contador)

322


### 1.2 Classificando as imagnes

In [1]:
import os
import random
import shutil
from PyPDF2 import PdfReader

pasta_imagens = "dataset-mias/"

com_nodulo = []
sem_nodulo = []

# LER PDF
reader = PdfReader("dataset-mias/00README.pdf")
linhas = []

for pagina in reader.pages:
    texto = pagina.extract_text()
    linhas.extend(texto.split("\n"))

# PROCESSAR
for linha in linhas:
    if "mdb" in linha:
        partes = linha.split()
        nome = partes[0] + ".pgm"
        
        if "NORM" in linha:
            sem_nodulo.append(nome)
        else:
            com_nodulo.append(nome)

# DIVIDIR
def dividir(lista, pasta_train, pasta_test):
    random.shuffle(lista)
    corte = int(0.7 * len(lista))
    
    for nome in lista[:corte]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_train, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)
    
    for nome in lista[corte:]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_test, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)

# PASTAS
os.makedirs("dataset/train/com_nodulo", exist_ok=True)
os.makedirs("dataset/train/sem_nodulo", exist_ok=True)
os.makedirs("dataset/test/com_nodulo", exist_ok=True)
os.makedirs("dataset/test/sem_nodulo", exist_ok=True)

# EXECUTAR
dividir(com_nodulo, "dataset/train/com_nodulo", "dataset/test/com_nodulo")
dividir(sem_nodulo, "dataset/train/sem_nodulo", "dataset/test/sem_nodulo")

print("Finalizado!")

Finalizado!


### 1.3 Carregando dataset para CNN

#### Data Augmentation (aumento de dados)

Agora para o modelo 4 vamos aumentar nosso dataset simplesmente efetuando rotações nas imagnes mamográficas.

Dessa forma, vamos aplicar Data Augmentation nos dados de treino e manter os dados de teste apenas redimensionado com a imagem limpa.

Além disso, foi ajustado a resolução da imagem para 128 x 128 e aplicado uma técnica chamada CLAHE, que aumenta o contraste da imagem.

In [15]:
import cv2
import numpy as np
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from collections import Counter

# 1. Defina a classe do CLAHE
class ApplyCLAHE(object):
    def __init__(self):
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    def __call__(self, img):
        img_np = np.array(img)

        # força grayscale
        if len(img_np.shape) == 3:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

        # normaliza corretamente para uint8
        if img_np.dtype != np.uint8:
            img_np = img_np.astype(np.float32)

            # normaliza entre 0 e 255
            img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
            img_np = (img_np * 255).astype(np.uint8)

        # aplica CLAHE
        img_clahe = self.clahe.apply(img_np)

        return Image.fromarray(img_clahe)

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    ApplyCLAHE(),  
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    ApplyCLAHE(),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(root="dataset/train", transform=train_transform)
test_dataset = datasets.ImageFolder(root="dataset/test", transform=test_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

## 2. Criando uma camada Wavelet usando PyTorch

#### WaveletLayer Treinável

Agora vamos permiti que a rede aprenda e melhore os pesos para a Wavelet automaticamente (self.weight = nn.Parameter(kernel)).

Além disso agora vamos usar os demais filtros:

- LL → estrutura geral
- LH → bordas horizontais
- HL → bordas verticais
- HH → detalhes finos

In [16]:
# Camada Wavelet 
import torch
import torch.nn as nn
import torch.nn.functional as F

class WaveletLayer(nn.Module):
    def __init__(self):
        super(WaveletLayer, self).__init__()

        # Filtros Haar (corretos)
        ll = torch.tensor([[0.5, 0.5],
                           [0.5, 0.5]])

        lh = torch.tensor([[-0.5, -0.5],
                           [0.5, 0.5]])

        hl = torch.tensor([[-0.5, 0.5],
                           [-0.5, 0.5]])

        hh = torch.tensor([[0.5, -0.5],
                           [-0.5, 0.5]])

        filtros = torch.stack([ll, lh, hl, hh], dim=0)  # (4, 2, 2)

        self.weight = nn.Parameter(filtros.unsqueeze(1), requires_grad=False)

    def forward(self, x):
        # aplica convolução com stride=2 (downsampling)
        x = F.conv2d(x, self.weight, stride=2)
        return x

## 3. Modelo com 3 convoluções de CNN integrando a camada Wavelet e com Data Augmentation

<img src="img/waveletCNN-modelo3.png" width="700" title="Dica de ferramenta">

In [17]:
"""
MODELO COM A WAVELET PARA REALCE E 2 CONVOLUÇÕES ("Wavelet Realce -> Conv -> Conv")
"""
class WaveletCNN(nn.Module):
    def __init__(self):
        super(WaveletCNN, self).__init__()

        self.wavelet = WaveletLayer()  # saída: 4 canais

        self.features = nn.Sequential(
            nn.Conv2d(4, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.wavelet(x)
        x = self.features(x)
        x = self.classifier(x)
        return x

#### Antes do treino:

In [18]:
from collections import Counter

labels_list = [label for _, label in train_dataset.samples]

class_counts_dict = Counter(labels_list)

print("Distribuição das classes:", class_counts_dict)

num_normal = class_counts_dict[0]
num_cancer = class_counts_dict[1]

Distribuição das classes: Counter({1: 143, 0: 80})


### 3.1 Treinando o modelo

#### Treinamento otimizado

Calcula a média de erro de todos os lotes e mostra a porcentagem de acertos enquanto o modelo aprende.

Aalém disso código já inclui o .to(device), o que acelera o treino em até 10x caso seja rodado em ambiente que tenha uma placa de vídeo ou estiver usando o Google Colab.

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Modelo
model = WaveletCNN()

# 2. Dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3. Loss com balanceamento
class_counts = [num_normal, num_cancer]

weights = 1. / torch.tensor(class_counts, dtype=torch.float)
weights = weights / weights.sum()

criterion = nn.CrossEntropyLoss(weight=weights.to(device))

# 4. Otimizador
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# 5. Scheduler (corrigido)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)


epochs = 50

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    corretos_treino = 0
    total_treino = 0
    
    for imagens, labels in train_loader:
        imagens, labels = imagens.to(device), labels.to(device)
        
        outputs = model(imagens)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_treino += labels.size(0)
        corretos_treino += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * corretos_treino / total_treino
    
    print(f"Época [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Acc Treino: {epoch_acc:.2f}%")

    scheduler.step(epoch_loss)

    # Validação
    if (epoch + 1) % 5 == 0:
        model.eval()
        corretos_teste = 0
        total_teste = 0

        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                out = model(imgs)
                _, pred = torch.max(out, 1)

                total_teste += lbls.size(0)
                corretos_teste += (pred == lbls).sum().item()

        print(f">>> Acurácia de Teste: {100 * corretos_teste / total_teste:.2f}%")

Época [1/50] - Loss: 0.7100 - Acc Treino: 46.64%
Época [2/50] - Loss: 0.7005 - Acc Treino: 55.16%
Época [3/50] - Loss: 0.7056 - Acc Treino: 48.88%
Época [4/50] - Loss: 0.6936 - Acc Treino: 58.74%
Época [5/50] - Loss: 0.6899 - Acc Treino: 50.67%
>>> Acurácia de Teste: 63.92%
Época [6/50] - Loss: 0.6800 - Acc Treino: 60.09%
Época [7/50] - Loss: 0.6907 - Acc Treino: 53.81%
Época [8/50] - Loss: 0.6780 - Acc Treino: 50.67%
Época [9/50] - Loss: 0.6743 - Acc Treino: 56.05%
Época [10/50] - Loss: 0.6843 - Acc Treino: 58.74%
>>> Acurácia de Teste: 54.64%
Época [11/50] - Loss: 0.6706 - Acc Treino: 57.85%
Época [12/50] - Loss: 0.6661 - Acc Treino: 58.74%
Época [13/50] - Loss: 0.6571 - Acc Treino: 57.40%
Época [14/50] - Loss: 0.6720 - Acc Treino: 61.88%
Época [15/50] - Loss: 0.6800 - Acc Treino: 58.30%
>>> Acurácia de Teste: 48.45%
Época [16/50] - Loss: 0.6469 - Acc Treino: 63.23%
Época [17/50] - Loss: 0.6601 - Acc Treino: 58.30%
Época [18/50] - Loss: 0.6850 - Acc Treino: 56.95%
Época [19/50] - Los

### 3.2 Testando o modelo

In [20]:
model.eval()
corretos = 0
total = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with torch.no_grad():
    for imagens, labels in test_loader:
        
        imagens, labels = imagens.to(device), labels.to(device)

        outputs = model(imagens)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        corretos += (predicted == labels).sum().item()

print(f"Acurácia: {100 * corretos / total:.2f}%")

Acurácia: 53.61%
